# 06 — Optical-X-ray reprocessing: FRED fit, calibration, simulation, free-beta

Report section 8, Phases 0-5. This is the interactive, per-outburst workflow:
manually select an outburst window, fit the Norris FRED, fetch ZTF, calibrate
`C_source` against fixed literature betas, simulate the optical curve, and
(optionally) fit beta freely. Results persist to `xrb_outbursts.json` across
sessions.

**Session workflow**: see `docs/xrb_outburst_inspector_guide_phase0.md` for the
full per-outburst loop (this notebook shows the same steps, calling into
`src/xrb_pipeline/reprocessing/` instead of inlining the logic).

In [ ]:
import sys
sys.path.insert(0, '../src')
import json
import numpy as np

from xrb_pipeline.utils import json_db
from xrb_pipeline.utils.fetch import fetch_maxi_lightcurve_multiband
from xrb_pipeline.reprocessing import manual_inspector as mi
from xrb_pipeline.reprocessing import optical_simulation as osim
from xrb_pipeline.reprocessing import free_beta_fitting as fbf

## Section 0 — Load / seed the JSON database

In [ ]:
JSON_PATH = "xrb_outbursts.json"

SOURCE_CATALOGUE = {
    "J1911+005": {"display_name": "Aql X-1", "compact_object": "NS", "lmxb": True,
                   "ztf_id": "ZTF18accedeu", "ztf_band": "r", "outbursts": []},
    "J0622-003": {"display_name": "1A 0620-00", "compact_object": "BH", "lmxb": True,
                   "ztf_id": "ZTF18acszddo", "ztf_band": "r", "outbursts": []},
}

db = json_db.load_json(JSON_PATH, SOURCE_CATALOGUE)
json_db.summarize(db)

## Phase 0 — Manual FRED fit

Pick a source and an outburst window (use `05_hardness_hid.ipynb`'s full-light-curve
plot, or `scripts/fetch_lightcurve.py`, to find one first).

In [ ]:
ACTIVE_SOURCE = "J1911+005"
X1, X2 = 57580.0, 57750.0
T_START_GUESS, T_RISE_GUESS, T_DECAY_GUESS = 57578.0, 2.0, 20.0

maxi_data = fetch_maxi_lightcurve_multiband(ACTIVE_SOURCE)
fit = mi.fit_outburst_fred(maxi_data['mjd'], maxi_data['flux_total'], maxi_data['flux_total_err'],
                            X1, X2, T_START_GUESS, T_RISE_GUESS, T_DECAY_GUESS)
print(fit)

In [ ]:
if fit['fit_ok']:
    db = mi.save_outburst_fit(db, ACTIVE_SOURCE, fit)
    json_db.save_json(db, JSON_PATH)
else:
    print("Fit did not converge -- adjust the guesses above and re-run.")

## Phase 1-2 — ZTF fetch, HR states, hard-state calibration

In [ ]:
OUTBURST_IDX = len(db[ACTIVE_SOURCE]["outbursts"]) - 1   # the one just fitted
outburst = db[ACTIVE_SOURCE]["outbursts"][OUTBURST_IDX]

maxi_window = mi.crop_maxi_to_window(maxi_data, outburst["x1"], outburst["x2"])
hr, state = mi.assign_hr_states(maxi_window)

ztf_df = mi.fetch_ztf_for_outburst(db[ACTIVE_SOURCE]["ztf_id"], outburst["x1"], outburst["x2"])
if ztf_df is None:
    print("No ZTF data in this window -- Phase 2 skipped.")
else:
    calib = mi.run_phase2_calibration(ztf_df, maxi_window, state, outburst)
    outburst.update(calib)
    json_db.save_json(db, JSON_PATH)
    print(calib)

## Phase 3 — Simulate optical + dual-axis plot

In [ ]:
if ztf_df is not None and outburst.get("C_A") is not None:
    sim = osim.simulate_optical(outburst, ztf_df)
    db = osim.save_rms_to_json(db, ACTIVE_SOURCE, OUTBURST_IDX, sim["RMS_A"], sim["RMS_B"])
    json_db.save_json(db, JSON_PATH)
    print(f"RMS_A={sim['RMS_A']:.5f}  RMS_B={sim['RMS_B']:.5f}")
else:
    print("Skipped -- needs a calibrated outburst (Phase 2) with ZTF coverage.")

## Phase 4 — Cross-source summary

Run once several outbursts have been through Phases 0-3.

In [ ]:
summary_df = osim.flatten_calibrated_outbursts(db)
print(f"{len(summary_df)} calibrated outbursts")
summary_df.head()

In [ ]:
if len(summary_df) > 0:
    osim.plot_c_source_bars(summary_df)
    osim.plot_rms_comparison(summary_df)

## Phase 5 — Free-beta fitting

Solve for beta and C_source simultaneously instead of assuming a fixed literature
value, and compare against the NS/BH literature anchors.

In [ ]:
if ztf_df is not None and outburst.get("C_A") is not None:
    maxi_state_df = fbf.build_state_labelled_maxi(ACTIVE_SOURCE, outburst["x1"], outburst["x2"])
    x, y, w, sig_c, cal_band = fbf.build_calibration_pairs(maxi_state_df, ztf_df)
    beta_fit = fbf.fit_beta_free(x, y, w)
    print(beta_fit)

    if beta_fit["beta_free"] is not None:
        compact_object = db[ACTIVE_SOURCE]["compact_object"]
        beta_lit, beta_lit_err = fbf.literature_anchor(compact_object)
        n_sigma = abs(beta_fit["beta_free"] - beta_lit) / np.sqrt(
            beta_fit["sigma_beta_free"] ** 2 + beta_lit_err ** 2)
        print(f"{compact_object} literature beta={beta_lit}+/-{beta_lit_err} "
              f"-> {n_sigma:.2f} sigma from beta_free")

        fbf.plot_loglog_fit(x, y, sig_c, cal_band, beta_fit,
                             outburst["C_A"], outburst["C_B"], title=ACTIVE_SOURCE)

        db = fbf.save_beta_free_to_json(db, ACTIVE_SOURCE, OUTBURST_IDX, beta_fit)
        json_db.save_json(db, JSON_PATH)
else:
    print("Skipped -- needs a calibrated outburst with ZTF coverage.")

## Cross-source beta comparison

Run any time to see everything saved so far.

In [ ]:
beta_summary = fbf.cross_source_beta_summary(db)
beta_summary